
## 00_setup — Project 2: GitHub Streaming Pipeline

Creates all required directories for the streaming pipeline.

New vs Project 1:
- checkpoints/  → Auto Loader tracks processed files here
- schemas/      → Auto Loader saves inferred schema here
- Bronze = JSON files (not Parquet)
- Silver = Delta table (same as Project 1)

Key concept: checkpoints make pipelines restartable.
If the pipeline crashes mid-run, it resumes from exactly
where it left off — no reprocessing, no data loss.


In [0]:
%python
# Define config variables inline
BASE_PATH = "/Volumes/workspace/default/github_streaming"

# ── Create the volume if it doesn't exist ────────────────────
# STUDY NOTE:
#   Unity Catalog volumes must be created before you can write to them.
#   CREATE VOLUME IF NOT EXISTS is idempotent (safe to run multiple times)
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.github_streaming
""")
print("✓ Volume workspace.default.github_streaming ready")
BRONZE_PATH = f"{BASE_PATH}/bronze/events"
SILVER_PATH = f"{BASE_PATH}/silver/events"
GOLD_PATH = f"{BASE_PATH}/gold/event_summary"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_to_silver"
SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze_schema"

# ── Create all directories ────────────────────────────────────
# STUDY NOTE:
#   We need more directories than Project 1 because
#   Auto Loader needs dedicated folders for:
#   1. checkpoint — tracks processed files
#   2. schema     — saves inferred schema
#   These must exist BEFORE the stream starts.

dirs = [
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    CHECKPOINT_PATH,
    SCHEMA_PATH,
]

for path in dirs:
    dbutils.fs.mkdirs(path)
    print(f"✓ Created: {path}")

In [0]:
%python

# ── Verify directory structure ────────────────────────────────
print("\nDirectory structure:")
base_listing = dbutils.fs.ls(BASE_PATH)

for folder in base_listing:
    print(f"\n  {folder.name}")
    try:
        sub = dbutils.fs.ls(folder.path)
        for item in sub:
            print(f"    └── {item.name}")
    except:
        print(f"    └── (empty)")

In [0]:
%python

# ── Print pipeline flow ───────────────────────────────────────
# STUDY NOTE:
#   This cell documents the streaming pipeline flow.
#   In production this would be in a README or wiki.
#   For learning, keeping it in the notebook means
#   you always have the explanation next to the code.

print("""
STREAMING PIPELINE FLOW
=======================

Step 1: 01_simulate_landing
  → Downloads GitHub Archive JSON files
  → Drops them into Bronze path (simulates real-time arrival)
  → Each file = 1 hour of GitHub events (~10,000+ events)

Step 2: 02_autoloader_silver  ← KEY NOTEBOOK
  → Auto Loader watches Bronze path
  → Detects NEW files automatically via checkpoint
  → Reads JSON → applies schema → writes to Silver Delta
  → If re-run: only processes files not yet seen

Step 3: 03_gold_aggregate
  → Reads Silver Delta table
  → Aggregates: top repos, event types, active users
  → Writes to Gold Delta via MERGE

Step 4: 04_stream_monitor
  → Inspects checkpoint to see processed files
  → Shows stream progress and lag
  → Educational: proves Auto Loader tracking works

CHECKPOINT CONCEPT:
  checkpoint/
    └── commits/          ← which files were committed
    └── offsets/          ← position in the stream
    └── sources/          ← file list already processed

  Delete checkpoint → Auto Loader reprocesses everything
  Keep checkpoint  → Auto Loader only reads NEW files
""")

In [0]:
%python

# ── Verify environment ────────────────────────────────────────
import pyspark
from delta.tables import DeltaTable

print(f"Spark version  : {spark.version}")
print(f"Delta Lake     : available ✓")

# Verify Auto Loader format is available
try:
    # This just checks the format exists, doesn't read anything
    test = spark.readStream.format("cloudFiles")
    print(f"Auto Loader    : available ✓")
except Exception as e:
    print(f"Auto Loader    : {e}")

print("\nSetup complete — ready for 01_simulate_landing ✓")